In [3]:
import pandas as pd
from scipy import signal
from scipy import interpolate as interp
import math
import numpy as np
import matplotlib.pyplot as plt
import os
from scipy.signal import butter, filtfilt
import glm_toolbox as glm
import signal_processing_toolbox as processing
import friction_toolbox as fric

In [4]:
rows = []
subjects = ["S01", "S02", "S03", "S04"]
blocks = ["L1", "L2", "L3", "L4", "S1", "S2", "S3", "S4"]

for s in subjects:
    # Nombre d'essais dépend du sujet
    if s in ["S01", "S02"]:
        n_trials = 4
    else:
        n_trials = 3
    for b in blocks:
        for t in range(1, n_trials + 1):
            trial = f"{t:03d}"
            glm_path = f"./Groupe4_LUNDI_APREM/Gr4_{s}_{b}_{trial}.glm"
            
            if not os.path.exists(glm_path):
                continue
            
            glm_df = glm.import_data(glm_path)
            baseline = range(0, 400)

            rows.append({
                "subject": s,
                "block":   b,
                "trial":   trial,
                # --- signaux bruts, pas encore filtrés ---
                "time":    glm_df.loc[:, 'time'].to_numpy(),
                "Fygl":    glm_df.loc[:, 'Fygl'].to_numpy(),
                "Fxgl":    glm_df.loc[:, 'Fxgl'].to_numpy(),
                "Fzgl":    glm_df.loc[:, 'Fzgl'].to_numpy(),
                "Fygr":    glm_df.loc[:, 'Fygr'].to_numpy(),
                "Fxgr":    glm_df.loc[:, 'Fxgr'].to_numpy(),
                "Fzgr":    glm_df.loc[:, 'Fzgr'].to_numpy(),
                "GF":      glm_df.loc[:, 'GF'].to_numpy(),
                "LowAcc_X": glm_df.loc[:, 'LowAcc_X'].to_numpy(),
                "LowAcc_Z": glm_df.loc[:, 'LowAcc_Z'].to_numpy(),
                "baseline": baseline,
            })

df_raw = pd.DataFrame(rows)

In [ ]:
# Initialiser les colonnes AVANT la boucle
for col in ["acc", "LF", "LFx", "LFz", "cycle_starts", "cycle_ends", "ipk", "dGF"]:
    df_raw[col] = np.empty(len(df_raw), dtype=object)

for idx, row in df_raw.iterrows():
    print(f"Traitement : {row['subject']} | {row['block']} | {row['trial']}")
    baseline = row["baseline"]
    
    # Normal Force exerted by the thumb
    NF_thumb = row["Fygl"] - np.nanmean(row["Fygl"][baseline])
    
    # Vertical Tangential Force exerted by the thumb
    TFx_thumb = row["Fxgl"] - np.nanmean(row["Fxgl"][baseline])
    
    # Horizontal Tangential Force exerted by the thumb  
    TFz_thumb = row["Fzgl"] - np.nanmean(row["Fzgl"][baseline])
    
    # Normal Force exerted by the index
    NF_index = -(row["Fygr"] - np.nanmean(row["Fygr"][baseline]))
    
    # Vertical Tangential Force exerted by the index
    TFx_index = row["Fxgr"] - np.nanmean(row["Fxgr"][baseline])
    
    # Horizontal Tangential Force exerted by the index
    TFz_index = row["Fzgr"] - np.nanmean(row["Fzgr"][baseline])

    #%% Get acceleration, LF and GF
    time  = row["time"]
    # Acceleration selon le bloc
    if row["block"] in ["L1", "L2", "L3", "L4"]:
        accX_raw = row["LowAcc_X"] * (-9.81)
        acc = accX_raw - np.mean(accX_raw[baseline])
    elif row["block"] in ["S1", "S2", "S3", "S4"]:
        accZ_raw = row["LowAcc_Z"] * (-9.81)
        acc = accZ_raw - np.mean(accZ_raw[baseline])

    # GF et LF
    GF  = row["GF"] - np.nanmean(row["GF"][baseline])
    LFx = TFx_thumb + TFx_index
    LFz = TFz_thumb + TFz_index
    LF  = np.hypot(LFx, LFz)

    # %%Filter data
    freqAcq=800 #Acquisition sampling frequency
    freqFiltAcc=4 #Low-pass filter cut-off frequency for acceleration signals
    freqFiltForces=30 #Low-pass filter cut-off frequency for force signals
    
    acc = processing.filter_signal(acc, fs = freqAcq, fc = freqFiltAcc)
    GF   = processing.filter_signal(GF,   fs = freqAcq, fc = freqFiltForces)
    LF   = processing.filter_signal(LF,   fs = freqAcq, fc = freqFiltForces)
    LFx  = processing.filter_signal(LFx,  fs = freqAcq, fc = freqFiltForces)
    LFz  = processing.filter_signal(LFz,  fs = freqAcq, fc = freqFiltForces)

    df_raw.at[idx, "acc"] = acc
    df_raw.at[idx, "GF"]  = GF 
    df_raw.at[idx, "LF"]  = LF
    df_raw.at[idx, "LFx"] = LFx
    df_raw.at[idx, "LFz"] = LFz
    

    #%% CUTTING THE TASK INTO SEGMENTS (your first task)
    # For an oscillation task, this is very easy as we just need to find 
    # acceleration peaks
    pk = signal.find_peaks(acc,  prominence=2, distance=1200)
    ipk = pk[0]
    cycle_starts = ipk[:-1]
    cycle_ends = ipk[1:]-1
    
    
    #%% Compute derivative of GF
    dGF = processing.derive(GF,800)

    df_raw.at[idx, "cycle_starts"] = cycle_starts
    df_raw.at[idx, "cycle_ends"]   = cycle_ends
    df_raw.at[idx, "ipk"]          = ipk
    df_raw.at[idx, "dGF"]          = dGF
    
    #%% Basic plot of the data
    fig = plt.figure(figsize = [15,7])
    ax  = fig.subplots(3,1)
            
    ax[0].plot(time, acc)
    ax[0].plot(time[ipk],acc[ipk], linestyle='', marker='o', 
            markerfacecolor='None', markeredgecolor='r')
    ax[0].set_ylabel("Acceleration [m/s^2]", fontsize=13)
    ax[0].set_title(f"GLM data - Sujet {row['subject']}, Bloc {row['block']}, Essai {row['trial']}", fontsize=14, fontweight="bold")
    ax[0].set_ylim([-4,4])
    #ax[0].set_xlim([0,60])
    
    # Putting grey patches for movement cycles
    for i in range(0,len(cycle_starts)):
        rect0=plt.Rectangle((time[cycle_starts[i]],ax[0].get_ylim()[0]),\
                        time[cycle_ends[i]-cycle_starts[i]],\
                        ax[0].get_ylim()[1]-ax[0].get_ylim()[0],color='k',alpha=0.3)
        ax[0].add_patch(rect0)
            
    ax[1].plot(time,LF, label="LF")
    ax[1].plot(time,GF, label="GF")
    ax[1].legend(fontsize=12)
    ax[1].set_xlabel("Time [s]", fontsize=13)
    ax[1].set_ylabel("Forces [N]", fontsize=13)
    #ax[1].set_xlim([0,30])
    
    ax[2].plot(time,dGF)
    ax[2].set_xlabel("Time [s]", fontsize=13)
    ax[2].set_ylabel("GF derivative [N/s]", fontsize=13)
    #ax[2].set_xlim([0,30])
    
    output_folder = f"figures/{row['subject']}"
    os.makedirs(output_folder, exist_ok=True)

    fig.savefig(f"{output_folder}/{row['subject']}_{row['block']}_{row['trial']}.png")
    plt.close(fig)


In [6]:
rows_friction = []

subjects = ["S01", "S02", "S03", "S04"]
trials = 3

for s in subjects:
    for t in range(1, trials + 1):
        trial = f"{t:03d}"
        glm_path = f"./Groupe4_LUNDI_APREM/Gr4_{s}_friction_{trial}.glm"
        
        if not os.path.exists(glm_path):
            print(f"❌ Fichier manquant : {glm_path}")
            continue
        
        glm_df = glm.import_data(glm_path)
        baseline = range(0, 400)

        rows_friction.append({
            "subject":  s,
            "trial":    trial,
            "time":     glm_df.loc[:, 'time'].to_numpy(),
            "Fygl":     glm_df.loc[:, 'Fygl'].to_numpy(),
            "Fxgl":     glm_df.loc[:, 'Fxgl'].to_numpy(),
            "Fzgl":     glm_df.loc[:, 'Fzgl'].to_numpy(),
            "Fygr":     glm_df.loc[:, 'Fygr'].to_numpy(),
            "Fxgr":     glm_df.loc[:, 'Fxgr'].to_numpy(),
            "Fzgr":     glm_df.loc[:, 'Fzgr'].to_numpy(),
            "baseline": baseline,
        })

df_friction = pd.DataFrame(rows_friction)


In [ ]:
freqAcq = 800
freqFiltForces = 20

# Initialiser les nouvelles colonnes
for col in ["NF_thumb", "TFx_thumb", "TFz_thumb", 
            "NF_index", "TFx_index", "TFz_index",
            "COPthumb", "COPindex",
            "mu_thumb", "mu_index",
            "slip_indexes_thumb", "slip_indexes_index",
            "start_search_zones_thumb", "end_search_zones_thumb",
            "start_search_zones_index", "end_search_zones_index"]:
    df_friction[col] = np.empty(len(df_friction), dtype=object)

# Colonnes k et n par sujet (scalaires, pas des arrays)
for col in ["k_thumb", "n_thumb", "k_index", "n_index"]:
    df_friction[col] = np.nan

for idx, row in df_friction.iterrows():
    print(f"Traitement : {row['subject']} | {row['trial']}")
    baseline = row["baseline"]

    # Calcul et filtrage des forces
    NF_thumb  = processing.filter_signal(row["Fygl"] - np.nanmean(row["Fygl"][baseline]), fs=freqAcq, fc=freqFiltForces)
    TFx_thumb = processing.filter_signal(row["Fxgl"] - np.nanmean(row["Fxgl"][baseline]), fs=freqAcq, fc=freqFiltForces)
    TFz_thumb = processing.filter_signal(row["Fzgl"] - np.nanmean(row["Fzgl"][baseline]), fs=freqAcq, fc=freqFiltForces)
    NF_index  = processing.filter_signal(-(row["Fygr"] - np.nanmean(row["Fygr"][baseline])), fs=freqAcq, fc=freqFiltForces)
    TFx_index = processing.filter_signal(row["Fxgr"] - np.nanmean(row["Fxgr"][baseline]), fs=freqAcq, fc=freqFiltForces)
    TFz_index = processing.filter_signal(row["Fzgr"] - np.nanmean(row["Fzgr"][baseline]), fs=freqAcq, fc=freqFiltForces)

    # COP — ici on a besoin du glm_df original donc on relit le fichier
    glm_path = f"./Groupe4_LUNDI_APREM/Gr4_{row['subject']}_friction_{row['trial']}.glm"
    glm_df = glm.import_data(glm_path)
    COPthumb, COPindex = glm.compute_cop(glm_df, baseline)
    COPthumb = processing.filter_signal(COPthumb, fs=freqAcq, fc=freqFiltForces)
    COPindex = processing.filter_signal(COPindex, fs=freqAcq, fc=freqFiltForces)

    # Slip onset
    mu_thumb, slip_idx_thumb, start_search_zones_thumb, end_search_zones_thumb = fric.find_slip_onset(COPthumb, TFx_thumb, TFz_thumb, NF_thumb)
    mu_index, slip_idx_index, start_search_zones_index, end_search_zones_index = fric.find_slip_onset(COPindex, TFx_index, TFz_index, NF_index)

    # Stockage
    df_friction.at[idx, "NF_thumb"]           = NF_thumb
    df_friction.at[idx, "TFx_thumb"]          = TFx_thumb
    df_friction.at[idx, "TFz_thumb"]          = TFz_thumb
    df_friction.at[idx, "NF_index"]           = NF_index
    df_friction.at[idx, "TFx_index"]          = TFx_index
    df_friction.at[idx, "TFz_index"]          = TFz_index
    df_friction.at[idx, "COPthumb"]           = COPthumb
    df_friction.at[idx, "COPindex"]           = COPindex
    df_friction.at[idx, "mu_thumb"]           = mu_thumb
    df_friction.at[idx, "mu_index"]           = mu_index
    df_friction.at[idx, "slip_indexes_thumb"] = slip_idx_thumb
    df_friction.at[idx, "slip_indexes_index"] = slip_idx_index
    df_friction.at[idx, "start_search_zones_thumb"] = start_search_zones_thumb
    df_friction.at[idx, "end_search_zones_thumb"]   = end_search_zones_thumb
    df_friction.at[idx, "start_search_zones_index"] = start_search_zones_index
    df_friction.at[idx, "end_search_zones_index"]   = end_search_zones_index

    
for idx, row in df_friction.iterrows():
    s     = row["subject"]
    trial = row["trial"]
    time  = row["time"]

    COPindex              = row["COPindex"]
    COPthumb              = row["COPthumb"]
    TFx_index             = row["TFx_index"]
    TFz_index             = row["TFz_index"]
    NF_index              = row["NF_index"]
    TFx_thumb             = row["TFx_thumb"]
    TFz_thumb             = row["TFz_thumb"]
    NF_thumb              = row["NF_thumb"]
    mu_index              = row["mu_index"]
    mu_thumb              = row["mu_thumb"]
    slip_indexes_index    = row["slip_indexes_index"]
    slip_indexes_thumb    = row["slip_indexes_thumb"]
    start_search_zones_index = row["start_search_zones_index"]
    end_search_zones_index   = row["end_search_zones_index"]
    start_search_zones_thumb = row["start_search_zones_thumb"]
    end_search_zones_thumb   = row["end_search_zones_thumb"]

    friction_folder = f"figures/{s}/friction"
    os.makedirs(friction_folder, exist_ok=True)

    #%% Graphe index
    siz = len(start_search_zones_index)
    fig = plt.figure(figsize=[15, 7])
    ax  = fig.subplots(3, 1)
    ax[0].set_title(f"{s}_friction_{trial} - Index", fontsize=14, fontweight="bold")
    ax[0].plot(time, COPindex * 1000)
    ax[0].set_ylabel("COP [mm]", fontsize=13)
    ax[0].set_ylim([-25, 25])
    
    ax[1].plot(time, TFx_index, label="TFv")
    ax[1].plot(time, NF_index, label="NF")
    ax[1].legend(fontsize=12)
    ax[1].set_ylabel("Forces [N]", fontsize=13)
    
    ax[2].plot(time, np.divide(np.hypot(TFx_index, TFz_index), NF_index))
    ax[2].plot(time[slip_indexes_index], mu_index, marker='.', linestyle='')
    ax[2].set_ylim([0, max(mu_index) + 0.2])
    ax[2].set_ylabel("TF/NF [-]", fontsize=13)
    ax[2].set_xlabel("Time [s]", fontsize=13)
    
    for i in range(siz):
        for ax_, ylim in zip([ax[0], ax[1], ax[2]], [ax[0].get_ylim(), ax[1].get_ylim(), ax[2].get_ylim()]):
            rect = plt.Rectangle((time[start_search_zones_index[i]], ylim[0]),
                                  time[end_search_zones_index[i] - start_search_zones_index[i]],
                                  ylim[1] - ylim[0], color='k', alpha=0.2)
            ax_.add_patch(rect)
    
    fig.savefig(f"{friction_folder}/{s}_{trial}_index.png")
    plt.close(fig)

    #%% Graphe thumb
    siz = len(start_search_zones_thumb)
    fig = plt.figure(figsize=[15, 7])
    ax  = fig.subplots(3, 1)
    ax[0].set_title(f"{s}_friction_{trial} - Thumb", fontsize=14, fontweight="bold")
    ax[0].plot(time, COPthumb * 1000)
    ax[0].set_ylabel("COP [mm]", fontsize=13)
    ax[0].set_ylim([-25, 25])
    
    ax[1].plot(time, TFx_thumb, label="TFv")
    ax[1].plot(time, NF_thumb, label="NF")
    ax[1].legend(fontsize=12)
    ax[1].set_ylabel("Forces [N]", fontsize=13)
    
    ax[2].plot(time, np.divide(np.hypot(TFx_thumb, TFz_thumb), NF_thumb))
    ax[2].plot(time[slip_indexes_thumb], mu_thumb, marker='.', linestyle='')
    ax[2].set_ylim([0, max(mu_thumb) + 0.2])
    ax[2].set_ylabel("TF/NF [-]", fontsize=13)
    ax[2].set_xlabel("Time [s]", fontsize=13)
    
    for i in range(siz):
        for ax_, ylim in zip([ax[0], ax[1], ax[2]], [ax[0].get_ylim(), ax[1].get_ylim(), ax[2].get_ylim()]):
            rect = plt.Rectangle((time[start_search_zones_thumb[i]], ylim[0]),
                                  time[end_search_zones_thumb[i] - start_search_zones_thumb[i]],
                                  ylim[1] - ylim[0], color='k', alpha=0.2)
            ax_.add_patch(rect)
    
    fig.savefig(f"{friction_folder}/{s}_{trial}_thumb.png")
    plt.close(fig)
# Calcul de k et n PAR SUJET après la boucle
for s in subjects:
    mask_s = df_friction["subject"] == s
    friction_folder = f"figures/{s}/friction"
    os.makedirs(friction_folder, exist_ok=True)
    
    # Agréger tous les mu et NF du sujet
    all_mu_thumb = np.concatenate(df_friction.loc[mask_s, "mu_thumb"].values)
    all_NF_thumb = np.concatenate([
        row["NF_thumb"][row["slip_indexes_thumb"]] 
        for _, row in df_friction[mask_s].iterrows()
    ])
    
    all_mu_index = np.concatenate(df_friction.loc[mask_s, "mu_index"].values)
    all_NF_index = np.concatenate([
        row["NF_index"][row["slip_indexes_index"]] 
        for _, row in df_friction[mask_s].iterrows()
    ])

    # Fit
    mask_valid = (all_NF_thumb > 0) & (all_mu_thumb > 0) & np.isfinite(all_NF_thumb) & np.isfinite(all_mu_thumb)
    k_thumb, n_thumb = fric.get_mu_fit(all_mu_thumb[mask_valid], all_NF_thumb[mask_valid])

    mask_valid = (all_NF_index > 0) & (all_mu_index > 0) & np.isfinite(all_NF_index) & np.isfinite(all_mu_index)
    k_index, n_index = fric.get_mu_fit(all_mu_index[mask_valid], all_NF_index[mask_valid])

    # Stocker k et n sur toutes les lignes du sujet
    df_friction.loc[mask_s, "k_thumb"] = k_thumb
    df_friction.loc[mask_s, "n_thumb"] = n_thumb
    df_friction.loc[mask_s, "k_index"] = k_index
    df_friction.loc[mask_s, "n_index"] = n_index

    #%% Figure finale pour l'index
    x = np.arange(0.2, 30, 0.02)
    fig = plt.figure(figsize=[15, 7])
    plt.plot(x, k_index * (x ** (n_index - 1)))
    plt.plot(all_NF_index, all_mu_index, linestyle='', marker='.')
    plt.ylim([0, 3])
    plt.xlim([0, 30])
    plt.title(f'Coefficient of friction index - {s}')
    plt.xlabel('Normal Force [N]')
    plt.ylabel('Static Friction [-]')
    fig.savefig(f"{friction_folder}/{s}_fit_index.png")
    plt.close(fig)

    #%% Figure finale pour le pouce
    fig = plt.figure(figsize=[15, 7])
    plt.plot(x, k_thumb * (x ** (n_thumb - 1)))
    plt.plot(all_NF_thumb, all_mu_thumb, linestyle='', marker='.')
    plt.ylim([0, 3])
    plt.xlim([0, 30])
    plt.title(f'Coefficient of friction thumb - {s}')
    plt.xlabel('Normal Force [N]')
    plt.ylabel('Static Friction [-]')
    fig.savefig(f"{friction_folder}/{s}_fit_thumb.png")
    plt.close(fig)

    print(f"{s} - Index: k={k_index:.4f}, n={n_index:.4f}")
    print(f"{s} - Thumb: k={k_thumb:.4f}, n={n_thumb:.4f}")


In [ ]:
# Initialiser les colonnes résultats
for col in ["SF_per_cycle", "SM_per_cycle", "GF_peak_per_cycle", "LF_peak_per_cycle"]:
    df_raw[col] = np.empty(len(df_raw), dtype=object)

for idx, row in df_raw.iterrows():
    s     = row["subject"]
    b     = row["block"]
    trial = row["trial"]
    print(f"Calcul SF/SM : {s} | {b} | {trial}")

    # Récupérer k et n du sujet depuis df_friction
    mask_s  = df_friction["subject"] == s
    k_thumb = df_friction.loc[mask_s, "k_thumb"].iloc[0]
    n_thumb = df_friction.loc[mask_s, "n_thumb"].iloc[0]
    k_index = df_friction.loc[mask_s, "k_index"].iloc[0]
    n_index = df_friction.loc[mask_s, "n_index"].iloc[0]
    
    # Moyenne k et n entre pouce et index
    k = (k_thumb + k_index) / 2
    n = (n_thumb + n_index) / 2

    GF           = row["GF"]
    LF           = row["LF"]
    cycle_starts = row["cycle_starts"]
    cycle_ends   = row["cycle_ends"]

    SF_list  = []
    SM_list  = []
    GF_peaks = []
    LF_peaks = []

    for i in range(len(cycle_starts)):
        GF_cycle = GF[cycle_starts[i]:cycle_ends[i]]
        LF_cycle = LF[cycle_starts[i]:cycle_ends[i]]

        # GF et LF au moment du pic de LF
        LF_peak_idx = np.argmax(LF_cycle)
        LF_peak     = LF_cycle[LF_peak_idx]
        GF_peak     = GF_cycle[LF_peak_idx]

        # Calcul SF et SM
        SF = (LF_peak / k) ** (1 / n)
        SM = (GF_peak - SF) / SF

        SF_list.append(SF)
        SM_list.append(SM)
        GF_peaks.append(GF_peak)
        LF_peaks.append(LF_peak)

    df_raw.at[idx, "SF_per_cycle"]      = np.array(SF_list)
    df_raw.at[idx, "SM_per_cycle"]      = np.array(SM_list)
    df_raw.at[idx, "GF_peak_per_cycle"] = np.array(GF_peaks)
    df_raw.at[idx, "LF_peak_per_cycle"] = np.array(LF_peaks)

In [ ]:
rows_summary = []

for idx, row in df_raw.iterrows():
    for i in range(len(row["SM_per_cycle"])):
        rows_summary.append({
            "subject": row["subject"],
            "block":   row["block"],
            "trial":   row["trial"],
            "cycle":   i,
            "SM":      row["SM_per_cycle"][i],
            "SF":      row["SF_per_cycle"][i],
            "GF_peak": row["GF_peak_per_cycle"][i],
            "LF_peak": row["LF_peak_per_cycle"][i],
        })

df_summary = pd.DataFrame(rows_summary)
print(df_summary.groupby(["subject", "block"])[["SM", "SF"]].mean())

In [ ]:
# ============================================================
# GRAPHE 1 — Boxplot SM par bloc (tous sujets confondus)
# ============================================================
output_folder_stats = "figures/stats"
os.makedirs(output_folder_stats, exist_ok=True)

fig, ax = plt.subplots(figsize=[15, 7])
blocks_order = ["L1", "L2", "L3", "L4", "S1", "S2", "S3", "S4"]
data_per_block = [df_summary[df_summary["block"] == b]["SM"].values for b in blocks_order]

ax.boxplot(data_per_block, labels=blocks_order)
ax.axhline(y=0, color='r', linestyle='--', label='SM = 0 (limite glissement)')
ax.set_xlabel("Bloc", fontsize=13)
ax.set_ylabel("Safety Margin [-]", fontsize=13)
ax.set_title("Safety Margin par bloc — tous sujets", fontsize=14, fontweight="bold")
ax.legend(fontsize=12)

fig.savefig(f"{output_folder_stats}/SM_par_bloc.png")
plt.close(fig)
print("✅ Graphe 1 sauvegardé")

# Un graphe par sujet aussi
for s in subjects:
    output_folder_s = f"figures/{s}/stats"
    os.makedirs(output_folder_s, exist_ok=True)
    
    df_s = df_summary[df_summary["subject"] == s]
    data_per_block = [df_s[df_s["block"] == b]["SM"].values for b in blocks_order]
    
    fig, ax = plt.subplots(figsize=[15, 7])
    ax.boxplot(data_per_block, labels=blocks_order)
    ax.axhline(y=0, color='r', linestyle='--', label='SM = 0 (limite glissement)')
    ax.set_xlabel("Bloc", fontsize=13)
    ax.set_ylabel("Safety Margin [-]", fontsize=13)
    ax.set_title(f"Safety Margin par bloc — {s}", fontsize=14, fontweight="bold")
    ax.legend(fontsize=12)
    
    fig.savefig(f"{output_folder_s}/{s}_SM_par_bloc.png")
    plt.close(fig)

print("✅ Graphes 1 par sujet sauvegardés")


# ============================================================
# GRAPHE 4 — Evolution de la SM cycle par cycle
# ============================================================

# Un graphe par sujet ET par bloc
for s in subjects:
    output_folder_s = f"figures/{s}/stats"
    os.makedirs(output_folder_s, exist_ok=True)
    
    fig, axes = plt.subplots(2, 4, figsize=[20, 10], sharey=True)
    fig.suptitle(f"Evolution SM par cycle — {s}", fontsize=14, fontweight="bold")
    axes = axes.flatten()
    
    for i, b in enumerate(blocks_order):
        df_sb = df_summary[(df_summary["subject"] == s) & (df_summary["block"] == b)]
        
        for t in df_sb["trial"].unique():
            df_sbt = df_sb[df_sb["trial"] == t]
            axes[i].plot(df_sbt["cycle"].values, df_sbt["SM"].values, 
                        marker='o', label=f"Essai {t}")
        
        axes[i].axhline(y=0, color='r', linestyle='--')
        axes[i].set_title(f"Bloc {b}", fontsize=12)
        axes[i].set_xlabel("Cycle", fontsize=11)
        axes[i].set_ylabel("SM [-]", fontsize=11)
        axes[i].legend(fontsize=9)
    
    fig.tight_layout()
    fig.savefig(f"{output_folder_s}/{s}_SM_evolution_cycles.png")
    plt.close(fig)

# Un graphe global tous sujets superposés par bloc
fig, axes = plt.subplots(2, 4, figsize=[20, 10], sharey=True)
fig.suptitle("Evolution SM par cycle — tous sujets", fontsize=14, fontweight="bold")
axes = axes.flatten()
colors = {"S01": "blue", "S02": "orange", "S03": "green", "S04": "red"}

for i, b in enumerate(blocks_order):
    for s in subjects:
        df_sb = df_summary[(df_summary["subject"] == s) & (df_summary["block"] == b)]
        # Moyenne par cycle sur tous les essais
        mean_per_cycle = df_sb.groupby("cycle")["SM"].mean()
        axes[i].plot(mean_per_cycle.index, mean_per_cycle.values,
                    marker='o', label=s, color=colors[s])
    
    axes[i].axhline(y=0, color='r', linestyle='--')
    axes[i].set_title(f"Bloc {b}", fontsize=12)
    axes[i].set_xlabel("Cycle", fontsize=11)
    axes[i].set_ylabel("SM [-]", fontsize=11)
    axes[i].legend(fontsize=9)

fig.tight_layout()
fig.savefig(f"{output_folder_stats}/SM_evolution_cycles_tous_sujets.png")
plt.close(fig)
print("✅ Graphes 4 sauvegardés")


# ============================================================
# GRAPHE 8 — GF peak vs LF peak avec courbe SF
# ============================================================

# Un graphe par sujet
for s in subjects:
    output_folder_s = f"figures/{s}/stats"
    os.makedirs(output_folder_s, exist_ok=True)

    # Récupérer k et n du sujet
    mask_s  = df_friction["subject"] == s
    k_thumb = df_friction.loc[mask_s, "k_thumb"].iloc[0]
    n_thumb = df_friction.loc[mask_s, "n_thumb"].iloc[0]
    k_index = df_friction.loc[mask_s, "k_index"].iloc[0]
    n_index = df_friction.loc[mask_s, "n_index"].iloc[0]
    k = (k_thumb + k_index) / 2
    n = (n_thumb + n_index) / 2

    df_s = df_summary[df_summary["subject"] == s]

    fig, ax = plt.subplots(figsize=[10, 8])
    
    # Points par bloc avec couleurs différentes
    cmap = plt.cm.get_cmap("tab10")
    for j, b in enumerate(blocks_order):
        df_sb = df_s[df_s["block"] == b]
        ax.scatter(df_sb["LF_peak"], df_sb["GF_peak"], 
                  label=f"Bloc {b}", color=cmap(j), alpha=0.7)
    
    # Courbe SF — GF minimum pour ne pas glisser
    LF_range = np.linspace(0.1, df_s["LF_peak"].max() * 1.1, 200)
    SF_curve = (LF_range / k) ** (1 / n)
    ax.plot(LF_range, SF_curve, 'r--', linewidth=2, label='Slip Force (SF)')
    
    ax.set_xlabel("LF peak [N]", fontsize=13)
    ax.set_ylabel("GF peak [N]", fontsize=13)
    ax.set_title(f"GF peak vs LF peak — {s}", fontsize=14, fontweight="bold")
    ax.legend(fontsize=11)
    
    fig.savefig(f"{output_folder_s}/{s}_GF_vs_LF_peak.png")
    plt.close(fig)

# Un graphe global tous sujets
fig, axes = plt.subplots(1, 4, figsize=[24, 6], sharey=False)
fig.suptitle("GF peak vs LF peak — tous sujets", fontsize=14, fontweight="bold")

for j, s in enumerate(subjects):
    mask_s  = df_friction["subject"] == s
    k_thumb = df_friction.loc[mask_s, "k_thumb"].iloc[0]
    n_thumb = df_friction.loc[mask_s, "n_thumb"].iloc[0]
    k_index = df_friction.loc[mask_s, "k_index"].iloc[0]
    n_index = df_friction.loc[mask_s, "n_index"].iloc[0]
    k = (k_thumb + k_index) / 2
    n = (n_thumb + n_index) / 2

    df_s = df_summary[df_summary["subject"] == s]
    
    for i, b in enumerate(blocks_order):
        df_sb = df_s[df_s["block"] == b]
        axes[j].scatter(df_sb["LF_peak"], df_sb["GF_peak"],
                       label=f"Bloc {b}", color=cmap(i), alpha=0.7)
    
    LF_range = np.linspace(0.1, df_s["LF_peak"].max() * 1.1, 200)
    SF_curve = (LF_range / k) ** (1 / n)
    axes[j].plot(LF_range, SF_curve, 'r--', linewidth=2, label='SF')
    
    axes[j].set_xlabel("LF peak [N]", fontsize=12)
    axes[j].set_ylabel("GF peak [N]", fontsize=12)
    axes[j].set_title(f"{s}", fontsize=13, fontweight="bold")
    axes[j].legend(fontsize=9)

fig.tight_layout()
fig.savefig(f"{output_folder_stats}/GF_vs_LF_peak_tous_sujets.png")
plt.close(fig)
print("✅ Graphes 8 sauvegardés")

In [ ]:
# ============================================================
# Scatter plot GF vs LF — à partir du premier cycle
# ============================================================

output_folder_stats = "figures/stats"
os.makedirs(output_folder_stats, exist_ok=True)

# Garder 1 point sur 20 pour la lisibilité
downsample = 20

# --- Un graphe par sujet ---
for s in subjects:
    output_folder_s = f"figures/{s}/stats"
    os.makedirs(output_folder_s, exist_ok=True)

    fig, axes = plt.subplots(2, 4, figsize=[20, 10], sharey=True, sharex=True)
    fig.suptitle(f"Corrélation GF vs LF — {s}", fontsize=14, fontweight="bold")
    axes = axes.flatten()

    for i, b in enumerate(blocks_order):
        df_sb = df_raw[(df_raw["subject"] == s) & (df_raw["block"] == b)]
        
        for _, row in df_sb.iterrows():
            # Commencer au début du premier cycle
            start = row["cycle_starts"][0]
            LF_crop = row["LF"][start::downsample]
            GF_crop = row["GF"][start::downsample]
            
            axes[i].scatter(LF_crop, GF_crop, alpha=0.3, s=2)
        
        axes[i].set_title(f"Bloc {b}", fontsize=12)
        axes[i].set_xlabel("LF [N]", fontsize=11)
        axes[i].set_ylabel("GF [N]", fontsize=11)

    fig.tight_layout()
    fig.savefig(f"{output_folder_s}/{s}_scatter_GF_LF.png")
    plt.close(fig)
    print(f"✅ {s} sauvegardé")

# --- Un graphe global tous sujets ---
fig, axes = plt.subplots(1, 4, figsize=[24, 6])
fig.suptitle("Corrélation GF vs LF — tous sujets", fontsize=14, fontweight="bold")

for j, s in enumerate(subjects):
    for i, b in enumerate(blocks_order):
        df_sb = df_raw[(df_raw["subject"] == s) & (df_raw["block"] == b)]
        
        for _, row in df_sb.iterrows():
            start   = row["cycle_starts"][0]
            LF_crop = row["LF"][start::downsample]
            GF_crop = row["GF"][start::downsample]
            
            axes[j].scatter(LF_crop, GF_crop,
                          alpha=0.2, s=2, color=cmap(i),
                          label=b if _ == df_sb.index[0] else "")
    
    axes[j].set_title(f"{s}", fontsize=13, fontweight="bold")
    axes[j].set_xlabel("LF [N]", fontsize=12)
    axes[j].set_ylabel("GF [N]", fontsize=12)
    axes[j].legend(fontsize=8, markerscale=5)

fig.tight_layout()
fig.savefig(f"{output_folder_stats}/scatter_GF_LF_tous_sujets.png")
plt.close(fig)
print("✅ Scatter plot global sauvegardé")

In [ ]:
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

# ============================================================
# Préparer les données de base
# ============================================================
df_stats = df_summary.groupby(["subject", "block"])["SM"].mean().reset_index()
df_stats.columns = ["subject", "block", "SM_mean"]

# ============================================================
# T-tests appariés
# ============================================================
print("="*60)
print("T-TESTS APPARIÉS")
print("="*60)

comparisons = [
    ("L1", "L2", "Effet yeux Longitudinal"),
    ("S1", "S2", "Effet yeux Sagittal"),
]

for b1, b2, label in comparisons:
    group1 = df_stats[df_stats["block"] == b1]["SM_mean"].values
    group2 = df_stats[df_stats["block"] == b2]["SM_mean"].values
    t_stat, p_val = stats.ttest_rel(group1, group2)
    
    print(f"\n{label}")
    print(f"  {b1} : moyenne = {group1.mean():.3f} ± {group1.std():.3f}")
    print(f"  {b2} : moyenne = {group2.mean():.3f} ± {group2.std():.3f}")
    print(f"  t = {t_stat:.3f}, p = {p_val:.3f}")
    if p_val < 0.05:
        print(f"  ✅ Différence significative (p < 0.05)")
    else:
        print(f"  ❌ Différence non significative (p = {p_val:.3f})")
    print(f"  ⚠️  Attention : basé sur seulement 4 sujets")

# ============================================================
# ANOVA — Effet du son 8D
# ============================================================
print("\n" + "="*60)
print("ANOVA — Effet du son 8D")
print("="*60)

for direction, blocs in [("Longitudinal", ["L2", "L3", "L4"]),
                          ("Sagittal",     ["S2", "S3", "S4"])]:
    
    groups = [df_stats[df_stats["block"] == b]["SM_mean"].values for b in blocs]
    f_stat, p_val = stats.f_oneway(*groups)
    
    print(f"\nDirection : {direction} ({', '.join(blocs)})")
    for b, g in zip(blocs, groups):
        print(f"  {b} : moyenne = {g.mean():.3f} ± {g.std():.3f}")
    print(f"  F = {f_stat:.3f}, p = {p_val:.3f}")
    if p_val < 0.05:
        print(f"  ✅ Effet significatif du son 8D")
    else:
        print(f"  ❌ Pas d'effet significatif (p = {p_val:.3f})")
    print(f"  ⚠️  Attention : basé sur seulement 4 sujets")

# ============================================================
# Rappel fiabilité
# ============================================================
print("\n" + "="*60)
print("RAPPEL SUR LA FIABILITÉ")
print("="*60)
print("""
  - Nombre de sujets : 4  (idéal : 15-20)
  - Puissance statistique : faible
  - p > 0.05 ne signifie pas absence d'effet
  - Ces résultats sont exploratoires et non généralisables
""")

In [ ]:
for idx, row in df_raw.iterrows():
    print(f"\n{row['subject']} | {row['block']} | {row['trial']}")
    time         = row["time"]
    cycle_starts = row["cycle_starts"]
    cycle_ends   = row["cycle_ends"]
    for i in range(len(cycle_starts)):
        t_start = time[cycle_starts[i]]
        t_end   = time[cycle_ends[i]]
        print(f"  Cycle {i} : {t_start:.2f}s → {t_end:.2f}s  (durée {t_end-t_start:.2f}s)")